# Project: fine-tune ResNet and build a tiny GPT

Two modern workhorses. Part A: take ResNet-18, trained on a million ImageNet photos, and adapt it to CIFAR-10 (transfer learning). Part B: build a small GPT (a decoder-only transformer) from the attention you wrote in DL12, and train it on Shakespeare. Use a GPU runtime; allow about 30 minutes.

**How to use this notebook:** run the cells from top to bottom (Shift+Enter). In each step, replace every `___` in the code cell, run it, then run the **Check** cell under it. Stuck? Expand the hidden **Solution** cell (click the ▸ arrow). For speed, turn on a GPU first: *Runtime → Change runtime type → T4 GPU*.

## Setup
Run this once.

In [ ]:
import os, urllib.request, math, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt
TEST = os.environ.get("NB_TEST") == "1"          # used only by the automatic tests; ignore it
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)
tf = T.Compose([T.Resize(96), T.ToTensor(), T.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))])
if TEST:
    train_set = torchvision.datasets.FakeData(64, (3, 32, 32), 10, transform=tf)
    test_set = torchvision.datasets.FakeData(64, (3, 32, 32), 10, transform=tf)
else:
    full = torchvision.datasets.CIFAR10("data", train=True, download=True, transform=tf)
    train_set = torch.utils.data.Subset(full, range(10000))
    test_set = torch.utils.data.Subset(torchvision.datasets.CIFAR10("data", train=False, download=True, transform=tf), range(2000))
train_loader = torch.utils.data.DataLoader(train_set, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128)

def accuracy(model, loader):
    model.eval(); c = n = 0
    with torch.no_grad():
        for x, y in loader:
            c += (model(x.to(device)).argmax(1).cpu() == y).sum().item(); n += len(y)
    return c / n

def train(model, params, epochs, lr):
    opt = torch.optim.Adam(params, lr=lr); model.train()
    for _ in range(epochs):
        for x, y in train_loader:
            loss = F.cross_entropy(model(x.to(device)), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step()

## Step 1: Part A: swap the head

ResNet-18's last layer (`fc`) predicts 1,000 ImageNet classes. Replace it with a fresh linear layer for 10 classes, keeping the same number of inputs.

In [ ]:
net = torchvision.models.resnet18(weights=None if TEST else "IMAGENET1K_V1")
net.fc = nn.Linear(___, 10)
net = net.to(device)
print(net.fc)

In [ ]:
# Check: run me after the cell above
assert net.fc.out_features == 10 and net.fc.in_features == 512
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
net = torchvision.models.resnet18(weights=None if TEST else "IMAGENET1K_V1")
net.fc = nn.Linear(net.fc.in_features, 10)
net = net.to(device)
print(net.fc)

## Step 2: Train only the head, then everything

First freeze the pretrained backbone and train only the new head (fast, safe). Then unfreeze everything and fine-tune with a small learning rate.

In [ ]:
for p in net.parameters():
    p.requires_grad = False
for p in net.fc.parameters():
    p.requires_grad = ___
train(net, net.fc.parameters(), epochs=1 if TEST else 2, lr=1e-3)
acc_head = accuracy(net, test_loader); print("head only:", acc_head)

for p in net.parameters():
    p.requires_grad = True
train(net, net.parameters(), epochs=1 if TEST else 2, lr=1e-4)
acc_full = accuracy(net, test_loader); print("fine-tuned:", acc_full)

In [ ]:
# Check: run me after the cell above
assert 0 <= acc_head <= 1 and 0 <= acc_full <= 1
if not TEST:
    assert acc_full > 0.85 and acc_full > acc_head, "Fine-tuning everything should reach over 85%"
print("Correct! Compare: your CNN from scratch needed 50,000 images to reach ~75%.")

In [ ]:
#@title Solution (click ▸ to show)
for p in net.parameters():
    p.requires_grad = False
for p in net.fc.parameters():
    p.requires_grad = True
train(net, net.fc.parameters(), epochs=1 if TEST else 2, lr=1e-3)
acc_head = accuracy(net, test_loader); print("head only:", acc_head)

for p in net.parameters():
    p.requires_grad = True
train(net, net.parameters(), epochs=1 if TEST else 2, lr=1e-4)
acc_full = accuracy(net, test_loader); print("fine-tuned:", acc_full)

## Step 3: Part B: causal self-attention

The heart of GPT. Project the input to queries, keys and values, compute softmax(QK^T / sqrt(d)) with a causal mask so no position sees the future, and mix the values.

In [ ]:
if TEST:
    text = "To be, or not to be, that is the question.\n" * 300
else:
    text = urllib.request.urlopen("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt").read().decode()
chars = sorted(set(text)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}; itos = dict(enumerate(chars))
data = torch.tensor([stoi[c] for c in text])
CTX, D, HEADS = 128, 192, 6

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(D, 3 * D); self.proj = nn.Linear(D, D)
        self.register_buffer("mask", torch.tril(torch.ones(CTX, CTX)).bool())
    def forward(self, x):
        B, Tn, _ = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q, k, v = [t.view(B, Tn, HEADS, D // HEADS).transpose(1, 2) for t in (q, k, v)]
        att = q @ k.transpose(-2, -1) / math.sqrt(D // HEADS)
        att = att.masked_fill(~self.mask[:Tn, :Tn], ___)
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).reshape(B, Tn, D)
        return self.proj(y)

In [ ]:
# Check: run me after the cell above
attn = CausalSelfAttention()
a = torch.randn(1, 10, D); b = a.clone(); b[0, 7:] = torch.randn(3, D)
assert torch.allclose(attn(a)[0, :7], attn(b)[0, :7], atol=1e-5), "Changing future tokens must not change earlier outputs"
print("Correct!")

In [ ]:
#@title Solution (click ▸ to show)
if TEST:
    text = "To be, or not to be, that is the question.\n" * 300
else:
    text = urllib.request.urlopen("https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt").read().decode()
chars = sorted(set(text)); V = len(chars); stoi = {c: i for i, c in enumerate(chars)}; itos = dict(enumerate(chars))
data = torch.tensor([stoi[c] for c in text])
CTX, D, HEADS = 128, 192, 6

class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(D, 3 * D); self.proj = nn.Linear(D, D)
        self.register_buffer("mask", torch.tril(torch.ones(CTX, CTX)).bool())
    def forward(self, x):
        B, Tn, _ = x.shape
        q, k, v = self.qkv(x).split(D, dim=2)
        q, k, v = [t.view(B, Tn, HEADS, D // HEADS).transpose(1, 2) for t in (q, k, v)]
        att = q @ k.transpose(-2, -1) / math.sqrt(D // HEADS)
        att = att.masked_fill(~self.mask[:Tn, :Tn], float("-inf"))
        att = F.softmax(att, dim=-1)
        y = (att @ v).transpose(1, 2).reshape(B, Tn, D)
        return self.proj(y)

## Step 4: Assemble GPT and train it

A transformer block is attention and an MLP, each wrapped as x = x + layer(norm(x)) (a residual connection). Stack four blocks between a token+position embedding and an output layer. Fill in the residual for the MLP.

In [ ]:
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.attn = CausalSelfAttention()
        self.mlp = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = ___
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, D), nn.Embedding(CTX, D)
        self.blocks = nn.Sequential(*[Block() for _ in range(4)])
        self.ln, self.head = nn.LayerNorm(D), nn.Linear(D, V)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        return self.head(self.ln(self.blocks(x)))

gpt = TinyGPT().to(device)
opt = torch.optim.AdamW(gpt.parameters(), lr=1e-3)
losses = []
for step in range(20 if TEST else 2500):
    s = torch.randint(0, len(data) - CTX - 1, (32,))
    x = torch.stack([data[i:i + CTX] for i in s]).to(device); y = torch.stack([data[i + 1:i + CTX + 1] for i in s]).to(device)
    loss = F.cross_entropy(gpt(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    if step % 500 == 0: print(step, round(loss.item(), 3))

idx = torch.tensor([[stoi[c] for c in "ROMEO:" if c in stoi] or [0]], device=device)
gpt.eval()
with torch.no_grad():
    for _ in range(300):
        p = F.softmax(gpt(idx[:, -CTX:])[0, -1] / 0.8, dim=-1)
        idx = torch.cat([idx, torch.multinomial(p, 1)[None]], dim=1)
print("".join(itos[i] for i in idx[0].tolist()))

In [ ]:
# Check: run me after the cell above
assert np.mean(losses[-5:]) < np.mean(losses[:5]), "Loss should fall"
if not TEST:
    assert np.mean(losses[-100:]) < 1.8, "Aim for a loss below 1.8"
print("Correct! Same architecture as ChatGPT's ancestors, just much smaller.")

In [ ]:
#@title Solution (click ▸ to show)
class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.attn = CausalSelfAttention()
        self.mlp = nn.Sequential(nn.Linear(D, 4 * D), nn.GELU(), nn.Linear(4 * D, D))
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

class TinyGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok, self.pos = nn.Embedding(V, D), nn.Embedding(CTX, D)
        self.blocks = nn.Sequential(*[Block() for _ in range(4)])
        self.ln, self.head = nn.LayerNorm(D), nn.Linear(D, V)
    def forward(self, idx):
        x = self.tok(idx) + self.pos(torch.arange(idx.shape[1], device=idx.device))
        return self.head(self.ln(self.blocks(x)))

gpt = TinyGPT().to(device)
opt = torch.optim.AdamW(gpt.parameters(), lr=1e-3)
losses = []
for step in range(20 if TEST else 2500):
    s = torch.randint(0, len(data) - CTX - 1, (32,))
    x = torch.stack([data[i:i + CTX] for i in s]).to(device); y = torch.stack([data[i + 1:i + CTX + 1] for i in s]).to(device)
    loss = F.cross_entropy(gpt(x).reshape(-1, V), y.reshape(-1))
    opt.zero_grad(); loss.backward(); opt.step(); losses.append(loss.item())
    if step % 500 == 0: print(step, round(loss.item(), 3))

idx = torch.tensor([[stoi[c] for c in "ROMEO:" if c in stoi] or [0]], device=device)
gpt.eval()
with torch.no_grad():
    for _ in range(300):
        p = F.softmax(gpt(idx[:, -CTX:])[0, -1] / 0.8, dim=-1)
        idx = torch.cat([idx, torch.multinomial(p, 1)[None]], dim=1)
print("".join(itos[i] for i in idx[0].tolist()))